# データ・AI活用実践（初級）第02回

本日から本格的なデータ分析が始まります．前回はRの基本的な書き方を勉強しましたが，当然ながらそれだけで自由自在にプログラミングできるようになるはずはありません．ソースコードのほとんどは事前に皆さんに提供していますので，それが何を意味しているか理解して，適切に変更できるようになることが目下の目標です．

-----------------------------

## 環境データ分析：　収集したデータの観察方法

### 中身のクイックルック

大気環境データの扱いを勉強します．Rを使って解析していくのですが，とりあえず配布したデータの中身をチェックしてみましょう．CSVというファイル形式ですが，中身は人間が読めるテキストですので，Excelはもちろんメモ帳でも開くことができます．google drive上でダブルクリックすれば，ExcelかGoogleスプレッドシートで開けるので，開いてみてください．

まず気が付くのは1行目が日本語であることです．プログラミングをやっていくうえでは厄介なことこの上ないです．したがって，1行目を飛ばしてデータを読み込むことが必要です．しかし，そのまま何もしなければ，気温の列に入っているデータが気温であることがわかりません．適切な名前を付けておかないと扱いにくいです．そこで，データを読み込むときに，こちらにとって扱いやすくてわかりやすい別名をつけてしまいましょう．わかりやすさ優先で日本語を使ってしまうと本末転倒なので，アルファベットにしましょう．

ちなみに，ダブルクリックしてCSVファイルの中身を見ているからと言って，そのデータをRプログラミング中に扱えるわけではありません．Rのソースコード中で機械に「データを読め」と命令しなければ何も起こりません．そこで，外部のファイルをプログラム中に利用できるようにするための準備を行います．


In [ ]:
library(tidyverse) #tidyverseというライブラリを使います

なんか怪しげなメッセージが出たと思います．errorsみたいな単語もあって不安になりますが，もう一度実行すれば消えると思います．
`tidyverse`という「ライブラリ」で定義されている関数を利用できるようにしました．「ライブラリ」ってなに？だと思いますが，ちゃんと説明します．前回皆さんは，Rの練習として四則演算や対数の計算を勉強しました．関数も勉強しましたね．「関数」と言われたときに，少なくとも小郷原はまっさきに三角関数を思い浮かべます（理系だから？）．Rで三角関数は使えるのでしょうか．

In [ ]:
sin(0)

In [ ]:
sin(pi/2)

使えますね！めでたし．．．ですが，ちょっと冷静に考えてみてください．三角関数は「関数」です．`()`に$\pi/2$を与えれば，1をreturnしてくれます．では誰がsin関数を定義したのでしょうか？
`
sin <- function(x){
  ...
}
`
なんていうソースコードを書いたわけではないですよね？世の中にはこういうケースが無数にあります．当然ですが，三角関数は世界共通です．日本人が定義しても，インド人が定義しても，火星人が定義しても，同じ関数でなければなりません．それならば，日本人とインド人が別々にソースコードを書くのはあほらしいですよね？車輪の再開発です．したがって，PythonでもCでもRでも，万人に共通する関数は誰かが作ってくれていて，我々はそれを利用することができるのです．ここで使用したsin関数は，誰かが作ってくれていて，我々が知らないところに保存されていて，意識せぬまま利用しているのです．では，プログラムの外部にあるCSVファイルをプログラムで利用できるようにする「関数」は無いのでしょうか？**あるんです**．これを使えるようにする呪文が，

`library(tidyverse)`

です．`tidyverse`というプログラム群の中に，CSVファイルを読み込む関数が定義されているのです．

ではまず，扱うCSVファイルを**持ってきましょう**．

In [ ]:
csvurl <- "https://www.cc.kyoto-su.ac.jp/~ogohara/lecture/DataAI_FirstCourse/DATA_20250825_1min.csv"
download.file(csvurl, destfile = "DATA_1min.csv") #DATA_1min.csvという名前のファイルとしてダウンロード

何も出力されなければ正解です．ダウンロードしただけですから．次に，プログラム中で扱えるようにするために「読んで」みましょう．

In [ ]:
df<-read_csv('DATA_1min.csv', locale=locale(encoding='Shift_JIS'), col_names=c('date','temp','rh','pre','spre','ws','wd','maxws','maxwd','irad'),skip=1)

どうでしょうか．なんかエラーのようなものが出る人もいるかもしれませんが，とりあえず進みましょう．CSVファイル名を指定しているところはいいでしょう．問題はその後です．`locale`オプションは文字のエンコーディングを指定しています．実は日本語を機械が解釈可能なように表す方法は複数あるのです．MacやLinuxはUTF-8が普通です．Windowsは`Shift-JIS`なことが多いです．今回小郷原は，windowsでデータファイルを作成したのでShift-JISにします．その次，`col_names`はcolumn namesのことで「列名」ですね．`c()`は配列のことですので，CSVファイルの中の各列を左から`date`，`temp`，．．．`irad`という列名にしてください，ということですね．`skip`は最初の1行を飛ばして読んで，ということです．

このようにして読み込んだCSVファイルの中身は，dfという変数に入っています．’Data Frame’だと思えばわかりやすいです．確認してみましょう．

In [ ]:
df

うおっ！！！すごくないですか．これだけでもかなりの情報量です．まず，最初に`1440 × 10`を出力されているので，全部で1,440行（row）10列（column）あることがわかりますね．date列を見ると，どうやら1分ごとにあります．1440データで1分ごと，ということは，全部で1日分でしょうか．さらに，全部で10列あることも読み取れます．一番上には，自分で指定した列名が既に記載されています．列名の下に表示されている`<chr>`は「この列の型はCharacter=文字列」という意味です．`<dbl>`は倍精度浮動小数点実数という意味なのですが，どういうことかはコンピュータサイエンスの教科書を見ていただくことにして，とても精度の高い実数，くらいの理解で結構です．

さあここから分析を進めていきたいと思うのですが，実は不都合が1つあるのです．それは「軸」です．皆さんお分かりの通り，このデータは時系列データです．神山ホールの屋上に設置されたPOTEKAという気象観測装置で観測されました．だから時間順になっていることに意味があるのです．一方，date列を見ると`<chr>`になっています．文字列とは単なる文字の並びのことであり，2025/8/25 0:01の次が2025/8/25 0:02である，という必然性はありません．我々はこれが暦の表示形式だとわかっているからしっくりくるだけで，機械からするとこれがバラバラになっていようとその善悪の判断は付きません．これが文字列の問題なのです．そこで，date列が単なる文字列ではなく日時データなんだ，と機械に教えることにしましょう．CSVデータを読み直します．

In [ ]:
df<-read_csv( 'DATA_1min.csv',locale=locale(encoding='Shift_JIS'), col_names=c('date','temp','rh','pre','spre','ws','wd','maxws','maxwd','irad'),skip=1,col_types=cols(date=col_datetime(format="%Y/%m/%d %H:%M")))

できました？読み込んだだけですから，まだ何も出てきません．

In [ ]:
df

さっきと比べてdate列がなんか変わりましたよね？？謎の型になったはずです．そして，日時の表示形式が妙に詳しくなって，なかったはずの「秒」が生まれました．

これでとりあえず，時系列データとしての体裁が整いました．（まだまだ問題はありますが，それは次節）

------------------
### 文字列のデータ

先ほどはdate列が文字列になっているのをボロカスに言いました．このデータファイルには他にも文字列があります．風向です．風向が北北西という言説は人間にはとてもわかりやすいですが，分析するときにはかなりやっかいです．8/25の平均風向ってどうやって計算すればいいのでしょうか．（北北西+北+東南東）÷3の演算結果は何でしょうか．機械にそんな計算できるでしょうか．文字列の風向は数値の風向に変換しなければ使えないのです！！

実を言いますと，風向には定義があります．気象庁が定義しているのです．それによると，風向は16方位で表現され，北から時計回りに22.5°ずつ決めれていますが，「北」は最後です．すなわち，最初は北北東，次が北東，次が東北東，次が東．．．．という感じで，最後が北です．

In [ ]:
wds=c('北北東', '北東', '東北東', '東', '東南東', '南東', '南南東', '南', '南南西', '南西', '西南西', '西', '西北西', '北西', '北北西', '北')

北北東が角度で言えば22.5°，北東が45°．．．で，北が360°ですから，風向の数値は以下の並びになりますね．

In [ ]:
wdf=1:16
wdf=(wdf*22.5)%%360

`wdf`がどんな数値になるかわかりますか？やってみましょう．

In [ ]:
wdf

まず最初のwdf=1:16は1から16まで1刻みの整数列を作りましょう，という意味です（別の書き方もあります）．1, 2, 3, …, 16です．そのようにして作成した1から16までの整数値の配列全体に22.5を掛け算しています．22.5, 45, 67.5, …, 360になりますね．そして最後の演算子%%とは，「剰余」です．%%360は，「360で割った時の余り」を意味します．したがって，360以外は変わりませんが，360だけは0になります．

では次に何をやるのでしょうか．次にやることは，wdが北北東だったら22.5，北東だったら45，という具合に，wdの値に従って異なる実数値を持った新しい列を生成することです．実を言えば，この作業は（初級）であることを考えれば結構難易度が高いです．1行1行上から`wd`列の値をチェックし，北北東だったら22.5，北東だったら45，という具合に新しい数値を決めて，最後の1440行目まで行くのですから，ループが必要ですね．さらにその中身，「北北東だったら22.5，北東だったら45，という具合に．．．」ってどうやればいいでしょうか．

In [ ]:
wds=='東'

これは論理型の配列です．論理型とはTrue/Falseの2値しか持たないような型です．1/0だと思えばいいですね．wdsが’東’に等しいところだけTRUEで，ほかはFALSEになっています．**イコール=は2つ必要**です．1つだけだと代入になってしまうからです．この時TRUEの位置（インデックス）を取得するには，次のようにします．

In [ ]:
which(wds=='東')

そうすると，’東’が角度で何°なのかすぐわかりますね．`wdf`の4番目を参照すればいいわけです．

In [ ]:
wdf[which(wds=='東')]

できました？90がでました？大事なことは，4という数字をソースコードに全く書いていないことです．こちらが「’東’は前から4番目」だと知る必要はないのです．それではいよいよ，ループを使いましょう．まずは練習です．

In [ ]:
for (s in wds){
  print(s)
}

`wds`の中身を1つずつ`s`に代入し，`print(s)`で表示しているわけです．したがって，forループの中では`wds`の要素に共通した処理を記述できます．

In [ ]:
for (s in wds){
  print(wdf[which(wds==s)])
}

ほらできた！！！では，次は本番です．ます数値の風向データを入れておく列を作成しておきます．

In [ ]:
df['wddeg'] = 0

さて一瞬でやりましょう．

In [ ]:
for (s in wds){
  wdval <- wdf[which(wds==s)]
  df[df$wd == s,'wddeg'] <- wdval
}

`s`には`wds`の中身が順番に入ります．’北北東’とか’南東’とかですね．ループの中ではまず，その時の`s`が`wds`の中で何番目にあるのか見つけます（`which`）．それに対応する風向の数値データを`wdf`から抽出し，`wdval`に代入します．次に，DataFrameである`df`のうち，`wd`列が`s`に等しい行だけTRUEで他はFALSEになっているような配列`df$wd == s`を使って，`wd`列が`s`に等しい行の`wddeg`列の値を`wdval`に変更しています．後は`wds`のすべての要素について同じことを繰り返すだけですね．

In [ ]:
df

----------------
### 数値データを簡単な図にする

データを取得したら数字をにらむだけではなくて，折れ線グラフやヒストグラムみたいな図にしたいですよね．ここまで前処理を頑張ってくれば，絵にするだけならサクッとできてしまいます．

> 全く意味が分からないソースコードでしょうが，今はそれでいいのです．分析結果が図としてサクサク出てこないと楽しくないじゃないですか．

In [ ]:
ggplot2::ggplot( data=df, aes(x=date,y=pre)) + ggplot2::geom_line()

速いでしょ？`date`列をちゃんと日時だと教えているからです．これを見ると気圧（`pre`）は午前9時くらいから急激に下がっていますね．では気温はどうでしょう．

In [ ]:
ggplot2::ggplot( data=df, aes(x=date,y=temp)) + ggplot2::geom_line(color='red')

基本的には朝になれば気温が上がり，昼過ぎぐらいにピークを迎えてじわじわ下がる，はずです．しかし，なんだか急激に下がってますね．それもそのはず，この日は京都市内に「記録的短時間大雨情報」が発表されて，１９０６年の統計開始以来，１時間雨量としては最も多くなりました．それ以外はよく知っている日変化ですね．その中でも微妙な振動が午前から午後の早い時間帯にかけて見られます．じゃあ風速はどうでしょう．

In [ ]:
ggplot2::ggplot( data=df, aes(x=date,y=ws)) + ggplot2::geom_line(color='blue')

あまり特徴がないですね．ひたすらギザギザしているだけですね．たぶん，比較的強風が吹いたときが大雨のタイミングなのではないでしょうか？最後に，頑張って作成した風向データを折れ線グラフにしてみます．

In [ ]:
ggplot2::ggplot( data=df, aes(x=date,y=wddeg)) + ggplot2::geom_line(color='green')

このように，風向が真北をまたいで変動するときは風向の数値が小さな値と大きな値を行き来するので，このような描画方法だととてもわかりにくいですね．どのように見やすい図を書いて，効率的に分析していくかは次回以降に勉強しましょう．本日はとりあえず，データを読み込んで，中身を確認し，まずいデータをまずくないデータに変換し，さくっと絵にしてみる，で終わりです．もし余裕があれば，`color`オプションの値を変えて，自分の好きな色の線に変えてみましょう！

--------------------
## 自由課題

1. `df`の中にある`maxws`は最大瞬間風速で，`maxwd`は最大瞬間風速を記録した時の風向を意味します．`maxwd`を数値に変換し`maxwddeg`という新しい列に代入しましょう．
2. 小郷原が授業中に配布する「昨日のデータ」を読み込んで，いろいろ図にしてみましょう．
